# Partie I — MLP sur Adult Income

## 01. Importation des bibliothèques

In [102]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import copy

##  02. Reproductibilité et device CPU/GPU
Cette cellule prépare les bases de notre modèle en remplissant deux objectifs principaux :

1. **Assurer la reproductibilité (`set_seed`)** : En fixant la "graine" (seed) pour Python, NumPy et PyTorch, nous garantissons que les processus aléatoires (comme l'initialisation des poids ou le mélange des données) seront exactement les mêmes à chaque exécution. Cela rend les résultats constants et comparables.
2. **Assigner le matériel de calcul (`device`)** : Le code détecte automatiquement si une carte graphique (GPU / CUDA) est disponible pour accélérer massivement les calculs. Si aucune n'est trouvée, il utilisera le processeur classique (CPU) par défaut.

In [35]:
SEED = 42 
def set_seed(seed = 42): 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 03. Chargement du dataset Adult Income

In [36]:
adult = fetch_openml(name="adult", version=2, as_frame=True)

X = adult.data.copy()
y = adult.target.copy()

print("Dimensions X :", X.shape)
print("Dimensions y :", y.shape)
X.head()

Dimensions X : (48842, 14)
Dimensions y : (48842,)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


## 04. Compréhension et analyse du dataset

In [37]:
print("Colonnes du dataset :")
print(X.columns.tolist())

print("\nTypes des variables :")
print(X.dtypes)

Colonnes du dataset :
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']

Types des variables :
age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
dtype: object


In [38]:
y.dtypes

CategoricalDtype(categories=['<=50K', '>50K'], ordered=False, categories_dtype=str)

In [40]:
display(X.info())

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
dtypes: category(8), int64(6)
memory usage: 2.6 MB


None

In [44]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Variables numériques :", numeric_features)
print("Nombre :", len(numeric_features))

print("\nVariables catégorielles :", categorical_features)
print("Nombre :", len(categorical_features))

Variables numériques : ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Nombre : 6

Variables catégorielles : ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
Nombre : 8


In [43]:
display(y.value_counts(normalize=True) * 100)

class
<=50K    76.071823
>50K     23.928177
Name: proportion, dtype: float64

In [39]:
print("\nDistribution de la variable cible :")
df = y.value_counts().rename_axis('class').reset_index(name='count')
df['percentage %'] = (df['count'] / df['count'].sum() * 100).round(2)
display(df)


Distribution de la variable cible :


,class,count,percentage %
0,<=50K,37155,76.07
1,>50K,11687,23.93


In [18]:
print("Valeurs manquantes classiques :")
print(X.isna().sum())

Valeurs manquantes classiques :
age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
dtype: int64


In [52]:
print(f"\nValeurs dupliquées : {X.duplicated().sum()}")
print(f"\nValeurs manquantes (y compris les '?') : {X.isnull().sum().sum()}")


Valeurs dupliquées : 57

Valeurs manquantes (y compris les '?') : 6465


In [53]:
display(X.describe(include='all'))

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
count,48842.000000,46043,4.884200e+04,48842,48842.000000,48842,46033,48842,48842,48842,48842.000000,48842.000000,48842.000000,47985
unique,NaN,8,NaN,16,NaN,7,14,6,5,2,NaN,NaN,NaN,41
top,NaN,Private,NaN,HS-grad,NaN,Married-civ-spouse,Prof-specialty,Husband,White,Male,NaN,NaN,NaN,United-States
freq,NaN,33906,NaN,15784,NaN,22379,6172,19716,41762,32650,NaN,NaN,NaN,43832
mean,38.643585,NaN,1.896641e+05,NaN,10.078089,NaN,NaN,NaN,NaN,NaN,1079.067626,87.502314,40.422382,NaN
std,13.710510,NaN,1.056040e+05,NaN,2.570973,NaN,NaN,NaN,NaN,NaN,7452.019058,403.004552,12.391444,NaN
min,17.000000,NaN,1.228500e+04,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,NaN
25%,28.000000,NaN,1.175505e+05,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN
50%,37.000000,NaN,1.781445e+05,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN
75%,48.000000,NaN,2.376420e+05,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,45.000000,NaN


### Distribution des classes
On regarde `y` qui contient la cible et permet de verifier l'equilibre des classes.
- ` remarque : On evite  ``X.value_counts()`` sur la matrice de features: sur un tableau tres large, cela peut exploser la memoire. `

In [54]:
# Ne pas appliquer value_counts sur X : sur une matrice tres large, cela peut provoquer un MemoryError.
display(y.value_counts(normalize=True) * 100)

class
<=50K    76.071823
>50K     23.928177
Name: proportion, dtype: float64

## 05. Nettoyage des données

In [56]:
print("Valeurs manquantes classiques :")
print(X.isna().sum())

print("\nRecherche des '?' dans les colonnes catégorielles :")
for col in categorical_features:
    count_question = (X[col].astype(str).str.strip() == "?").sum()
    print(col, ":", count_question)

Valeurs manquantes classiques :
age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
dtype: int64

Recherche des '?' dans les colonnes catégorielles :
workclass : 0
education : 0
marital-status : 0
occupation : 0
relationship : 0
race : 0
sex : 0
native-country : 0


In [57]:
df = X.copy()
df["income"] = y

# Supprimer les lignes contenant des valeurs manquantes
df_clean = df.dropna().copy()

# Nettoyer les espaces seulement après suppression des NaN
for col in df_clean.select_dtypes(include=["object", "category"]).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print("Dimensions avant nettoyage :", df.shape)
print("Dimensions après nettoyage :", df_clean.shape)

print("\nValeurs manquantes après nettoyage :")
print(df_clean.isna().sum())


Dimensions avant nettoyage : (48842, 15)
Dimensions après nettoyage : (45222, 15)

Valeurs manquantes après nettoyage :
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


### Encodage de la cible :

In [67]:
print("\nValeurs uniques dans la variable cible :")
print(df_clean["income"].unique().tolist())



Valeurs uniques dans la variable cible :
['<=50K', '>50K']


In [68]:
print("\nDistribution de la variable cible après nettoyage :")
display(pd.DataFrame(df_clean["income"].value_counts()))

df_clean["target"] = df_clean["income"].map({
    "<=50K": 0,
    ">50K": 1
})
print("\nDistribution de la variable cible après encodage :")
display(pd.DataFrame(df_clean["target"].value_counts()))


Distribution de la variable cible après nettoyage :



Distribution de la variable cible après nettoyage :


,count
income,
<=50K,34014
>50K,11208



Distribution de la variable cible après nettoyage :


,count
income,
<=50K,34014
>50K,11208



Distribution de la variable cible après encodage :


,count
target,
0,34014
1,11208


### Séparation features / target :

In [70]:
print(df_clean.columns.tolist())
df_clean.head()

['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income', 'target']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,target
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K,1
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K,0


In [73]:
X_clean = df_clean.drop(columns=["income", "target"])
y_clean = df_clean["target"].astype(int)

numeric_features = X_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_clean.select_dtypes(include=["object", "category","string"]).columns.tolist()

display("Variables numériques :", numeric_features)
display("Variables catégorielles :", categorical_features)

'Variables numériques :'

'Variables numériques :'

['age',
 'fnlwgt',
 'education-num',
 'capital-gain',
 'capital-loss',
 'hours-per-week']

'Variables catégorielles :'

['workclass',
 'education',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country']

## 06. Split train / validation / test

`Important : on fait le split avant d’entraîner le préprocesseur pour éviter la fuite de données.`

In [76]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_clean,
    y_clean,
    test_size=0.30,
    random_state=SEED,
    stratify=y_clean
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Train :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

Train : (31655, 14) (31655,)
Validation : (6783, 14) (6783,)
Test : (6784, 14) (6784,)


In [84]:
print("Distribution train ( % ) :")
display(y_train.value_counts(normalize=True) * 100 )

print("\nDistribution validation ( % ) :")
display(y_val.value_counts(normalize=True) * 100 )

print("\nDistribution test ( % ) :")
display(y_test.value_counts(normalize=True) * 100 )

Distribution train ( % ) :


Distribution train ( % ) :


target
0    75.214026
1    24.785974
Name: proportion, dtype: float64


Distribution validation ( % ) :


target
0    75.217455
1    24.782545
Name: proportion, dtype: float64


Distribution test ( % ) :


target
0    75.221108
1    24.778892
Name: proportion, dtype: float64

## 07. Prétraitement : encodage et normalisation

version avec normalisation :

On va standardiser les variables numériques, encoder les variables textuelles et applique ces changements sur nos ensembles d'entraînement, de validation et de test.

In [94]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    print("Warning: 'sparse_output' parameter not available. Using 'sparse=False' instead.")
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", onehot, categorical_features)
    ]
)

X_train_scaled = preprocessor_scaled.fit_transform(X_train)
X_val_scaled = preprocessor_scaled.transform(X_val)
X_test_scaled = preprocessor_scaled.transform(X_test)

print("X_train après prétraitement :", X_train_scaled.shape)
display(pd.DataFrame(X_train_scaled).head())


X_train après prétraitement : (31655, 104)


,0,1,2,3,4,5,6,7,8,9,...,94,95,96,97,98,99,100,101,102,103
0,0.336395,-0.516254,1.510203,-0.148057,-0.2184,0.747279,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,1.770641,0.524995,1.120307,13.081812,-0.2184,-0.084400,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-1.324310,0.808359,-0.049379,-0.148057,-0.2184,-0.084400,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,-0.493957,0.117601,-1.219065,-0.148057,-0.2184,-0.084400,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.946877,-0.439026,-0.049379,-0.148057,-0.2184,-0.084400,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


Version sans normalisation, pour l’expérience comparative :

In [86]:
try:
    onehot_no_scale = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    print("Warning: 'sparse_output' parameter not available. Using 'sparse=False' instead.")
    onehot_no_scale = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor_no_scale = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", onehot_no_scale, categorical_features)
    ]
)

X_train_no_scale = preprocessor_no_scale.fit_transform(X_train)
X_val_no_scale = preprocessor_no_scale.transform(X_val)
X_test_no_scale = preprocessor_no_scale.transform(X_test)

print("X_train sans normalisation :", X_train_no_scale.shape)

X_train sans normalisation : (31655, 104)


## 08. Création des Dataset et DataLoader PyTorch

### 🧠 Création du Dataset PyTorch personnalisé

Cette cellule définit une classe personnalisée appelée `AdultIncomeDataset`. Elle hérite de la classe `Dataset` de **PyTorch**. 
Son but est de convertir nos tableaux de données (NumPy/Pandas) en objets compatibles avec les outils d'entraînement de PyTorch (comme les `DataLoader`).

La classe surcharge les **trois méthodes indispensables** imposées par PyTorch :

1. **`__init__(self, X, y)` (Initialisation) :**
   * Reçoit les fonctionnalités de prédiction (`X`) et la variable cible (`y`).
   * Convertit les données en **Tenseurs PyTorch** de type décimal (`torch.float32`).
   * Utilise `.view(-1, 1)` sur `y` pour s'assurer que la variable cible a la forme d'une colonne verticale (vecteur 2D), ce qui est obligatoire pour calculer les fonctions de perte (*loss*) en classification binaire.

2. **`__len__(self)` (Taille) :**
   * Renvoie le nombre total d'exemples (lignes) présents dans le jeu de données. 
   * Permet à PyTorch de savoir quand une époque d'entraînement est terminée.

3. **`__getitem__(self, idx)` (Extraction) :**
   * Permet de récupérer un exemple précis (`X[idx]`) associé à son étiquette (`y[idx]`) via son index.
   * Cette méthode est appelée automatiquement en arrière-plan pour charger les données par petits groupes (mini-batches).


In [87]:
class AdultIncomeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.to_numpy(), dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

### 📦 Préparation des chargeurs de données (DataLoader) et taille d'entrée

Cette cellule prépare l'alimentation du modèle PyTorch en découpant les données et en configurant la structure du futur réseau de neurones.

#### 1. Configuration des Mini-Batches
* **`BATCH_SIZE = 128`** : Définit la taille des lots. Le modèle ne va pas analyser tout le jeu de données d'un coup, mais par groupes de 128 lignes. Cela stabilise l'entraînement et évite de saturer la mémoire (RAM).

#### 2. Instanciation des Datasets
* Les données d'entraînement (`scaled`), de validation et de test sont converties en objets PyTorch en utilisant la classe personnalisée `AdultIncomeDataset` créée à l'étape précédente.

#### 3. Création des `DataLoader`
Le `DataLoader` gère la logique d'envoi des données au modèle durant l'apprentissage :
* **`train_loader` (`shuffle=True`)** : Mélange aléatoirement les données d'entraînement à chaque époque. C'est indispensable pour que le modèle n'apprenne pas l'ordre des lignes par cœur et généralise mieux.
* **`val_loader` & `test_loader` (`shuffle=False`)** : Ne mélangent pas les données. Pour l'évaluation, l'ordre n'a pas d'importance, désactiver le mélange permet de gagner du temps de calcul.

#### 4. Dimension d'entrée du réseau (`input_dim`)
* **`X_train_scaled.shape[1]`** : Récupère le nombre exact de colonnes finales après notre étape de prétraitement (StandardScaler + OneHotEncoder). 
* Cette variable `input_dim` est cruciale car elle dictera la taille de la **toute première couche** de votre réseau de neurones.


In [88]:
BATCH_SIZE = 128

train_dataset = AdultIncomeDataset(X_train_scaled, y_train)
val_dataset = AdultIncomeDataset(X_val_scaled, y_val)
test_dataset = AdultIncomeDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

input_dim = X_train_scaled.shape[1]
print("Nombre de features après encodage :", input_dim)

Nombre de features après encodage : 104


DataLoaders sans normalisation :

In [89]:
train_dataset_no_scale = AdultIncomeDataset(X_train_no_scale, y_train)
val_dataset_no_scale = AdultIncomeDataset(X_val_no_scale, y_val)
test_dataset_no_scale = AdultIncomeDataset(X_test_no_scale, y_test)

train_loader_no_scale = DataLoader(train_dataset_no_scale, batch_size=BATCH_SIZE, shuffle=True)
val_loader_no_scale = DataLoader(val_dataset_no_scale, batch_size=BATCH_SIZE, shuffle=False)
test_loader_no_scale = DataLoader(test_dataset_no_scale, batch_size=BATCH_SIZE, shuffle=False)

In [97]:
# Vérification d'un batch PyTorch
X_batch, y_batch = next(iter(train_loader))

print("X_batch shape :", X_batch.shape)
print("y_batch shape :", y_batch.shape)
print("X_batch dtype :", X_batch.dtype)
print("y_batch dtype :", y_batch.dtype)

assert X_batch.shape[1] == input_dim
assert y_batch.shape[1] == 1
assert X_batch.dtype == torch.float32
assert y_batch.dtype == torch.float32

print("Vérification Dataset/DataLoader réussie.")

X_batch shape : torch.Size([128, 104])
y_batch shape : torch.Size([128, 1])
X_batch dtype : torch.float32
y_batch dtype : torch.float32
Vérification Dataset/DataLoader réussie.


## 09. Définition des modèles MLP
On a deux methodes pour construire notre réseau de neurone
- nn.Sequential permet d’enchaîner les couches simplement et rapidement (pas compatible pour les architectures complexes)
- une classe personnalisée héritant de nn.Module offre plus de flexibilité avec une méthode forward().
### Version nn.Sequential

In [98]:
def build_mlp_sequential(input_dim):
    model = nn.Sequential(
        nn.Linear(input_dim, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 1)
    )
    return model

### Version classe personnalisée

In [99]:
class MLPCustom(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)

        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        logits = self.output(x)
        return logits

## 10. Initialisation des poids

Pour initialiser les poids, on a 3 strategies : 
- **Gaussienne** (normal) : poids petits tirer au hasard selon loi normale ( courbe de Gausse), apprentissage généralement stable.
- **Constante** : mauvaise en pratique, car les neurones commencent de manière identique, ce qui crée un problème de symétrie.
- **Xavier** : cherche à stabiliser la variance du signal pendant la propagation, on calculons l'écart des valeurs aléatoires en fonction du nombre de neurones d'entrée et de sortie de la couche. Plus une couche a de neurones, plus les poids initiaux seront petits. Cela évite que les signaux mathématiques ne deviennent trop gigantesques (exploding gradients) ou minuscules (vanishing gradients) au début de l'entraînement. 


In [100]:
def init_weights(model, strategy="xavier"):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            if strategy == "normal":
                nn.init.normal_(module.weight, mean=0.0, std=0.01)
                nn.init.zeros_(module.bias)

            elif strategy == "constant":
                nn.init.constant_(module.weight, 1.0)
                nn.init.zeros_(module.bias)

            elif strategy == "xavier":
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

            else:
                raise ValueError("Stratégie inconnue. Choisir : normal, constant ou xavier.")

## 11. Fonctions d'entraînement et d'évaluation

In [101]:
# Cette fonction prend les logits (sortie brute du modèle) et les étiquettes vraies, et calcule l'accuracy.
def compute_accuracy_from_logits(logits, y_true):
    probs = torch.sigmoid(logits) # transformez les logits en probabilités
    preds = (probs >= 0.5).float() # convertissez les probabilités en prédictions binaires 0 ou 1
    correct = (preds == y_true).sum().item()
    total = y_true.size(0) 
    return correct / total

In [103]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    running_acc = 0.0
    total_samples = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad() # réinitialiser les gradients pour ne pas accumuler les erreurs du batch précédent

        logits = model(X_batch)
        loss = criterion(logits, y_batch) # calcule l'erreur du modèle

        loss.backward()
        optimizer.step()

        batch_size = X_batch.size(0)
        running_loss += loss.item() * batch_size # loss.item() donne la valeur scalaire du loss pour le batch, on multiplie par batch_size pour obtenir la contribution totale du batch à la perte de l'époque
        running_acc += compute_accuracy_from_logits(logits, y_batch) * batch_size
        total_samples += batch_size

    epoch_loss = running_loss / total_samples
    epoch_acc = running_acc / total_samples

    return epoch_loss, epoch_acc

In [104]:
def evaluate(model, loader, criterion, device):
    # mettre le modèle en mode évaluation pour désactiver les comportements spécifiques à l'entraînement (ex: dropout)
    # on utilise 100% des neurones pour faire les prédictions, ce qui donne une évaluation plus précise des performances du modèle sur les données de validation ou de test
    model.eval()

    running_loss = 0.0
    running_acc = 0.0
    total_samples = 0
    
    #Puisque on veut juste lire le score pour evaluer et pas à modifier les neurones 
    #on va bloquer le calcule de gradients pour économiser de la mémoire et accélérer les calculs
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            batch_size = X_batch.size(0)
            #loss.item() est la moyenne du loss pour le batch, 
            #on multiplie par batch_size pour obtenir la contribution totale du batch à la perte de l'époque
            # d un autre terme on accumule la perte et l'accuracy pour tous les batches
            running_loss += loss.item() * batch_size
            running_acc += compute_accuracy_from_logits(logits, y_batch) * batch_size
            total_samples += batch_size

    epoch_loss = running_loss / total_samples
    epoch_acc = running_acc / total_samples

    return epoch_loss, epoch_acc

In [105]:
def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, device="cpu"):
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss() # fct de perte  pour classification binaire, combine un Sigmoid et un Binary Cross Entropy Loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr) # reçoit les paramètres du modèle et le taux d'apprentissage et adapte automatiquement la vitesse de réglafz pour chaque neurone

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_val_loss = float("inf")
    best_state_dict = None

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state_dict)
    return model, history

## 12. Expériences comparatives

In [106]:
EPOCHS = 20
LR = 1e-3

experiments = []
histories = {}
models = {}

### Expérience 1 — MLP Sequential + Xavier + normalisation

In [107]:
set_seed(SEED)

model_seq_xavier = build_mlp_sequential(input_dim)
init_weights(model_seq_xavier, strategy="xavier")

model_seq_xavier, history_seq_xavier = train_model(
    model_seq_xavier,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device
)

histories["sequential_xavier_scaled"] = history_seq_xavier
models["sequential_xavier_scaled"] = model_seq_xavier

Epoch 01/20 | Train Loss: 0.3644 | Val Loss: 0.3288 | Train Acc: 0.8260 | Val Acc: 0.8389
Epoch 02/20 | Train Loss: 0.3225 | Val Loss: 0.3219 | Train Acc: 0.8519 | Val Acc: 0.8417
Epoch 03/20 | Train Loss: 0.3158 | Val Loss: 0.3220 | Train Acc: 0.8538 | Val Acc: 0.8462
Epoch 04/20 | Train Loss: 0.3133 | Val Loss: 0.3208 | Train Acc: 0.8542 | Val Acc: 0.8455
Epoch 05/20 | Train Loss: 0.3109 | Val Loss: 0.3206 | Train Acc: 0.8557 | Val Acc: 0.8459
Epoch 06/20 | Train Loss: 0.3082 | Val Loss: 0.3221 | Train Acc: 0.8597 | Val Acc: 0.8453
Epoch 07/20 | Train Loss: 0.3067 | Val Loss: 0.3220 | Train Acc: 0.8588 | Val Acc: 0.8459
Epoch 08/20 | Train Loss: 0.3048 | Val Loss: 0.3207 | Train Acc: 0.8583 | Val Acc: 0.8458
Epoch 09/20 | Train Loss: 0.3014 | Val Loss: 0.3210 | Train Acc: 0.8603 | Val Acc: 0.8453
Epoch 10/20 | Train Loss: 0.2998 | Val Loss: 0.3230 | Train Acc: 0.8616 | Val Acc: 0.8449
Epoch 11/20 | Train Loss: 0.2990 | Val Loss: 0.3235 | Train Acc: 0.8626 | Val Acc: 0.8462
Epoch 12/2

### Interprétation des résultats de l'entraînement pour l'expérience 1 — MLP Sequential + Xavier + normalisation

L'analyse de l'évolution des métriques (Loss et Accuracy) au fil des 20 époques permet de dresser le diagnostic suivant :

#### 1. Diagnostic global : Début de surapprentissage (*Overfitting*) léger
Le modèle présente un comportement classique où l'apprentissage se déroule parfaitement dans un premier temps, avant de commencer à mémoriser le bruit des données :
* **Évolution de la Perte (*Loss*) :** La `Train Loss` diminue de manière fluide et continue de `0.3644` à `0.2841`. Cependant, la `Val Loss` atteint son point le plus bas à l'**Époque 05 (`0.3206`)**, avant de stagner puis de remonter légèrement pour finir à `0.3295`. 
* **Évolution de la Précision (*Accuracy*) :** La `Train Acc` progresse régulièrement pour atteindre `86.86%`. En contrepartie, la `Val Acc` sature et plafonne très tôt (dès l'Époque 03) autour d'un plateau situé entre **84.5% et 84.7%**.

#### 2. Rôle du point de sauvegarde (*Checkpointing*)
Grâce à l'utilisation de `copy.deepcopy(model.state_dict())` dans notre fonction de gestion, le modèle final retenu ne sera pas celui de la dernière époque (Époque 20). 

Le script restaure automatiquement l'état de l'**Époque 05**, qui offre le meilleur compromis et la perte de validation minimale.
* **Score final de référence (Validation) :** ~84.59% d'exactitude.

#### 3. Conclusions et pistes pour les expériences comparatives
* **Points positifs :** Un score de ~84.6% sur ce jeu de données est une excellente base de départ. L'apprentissage est stable dès la première époque, ce qui valide la stratégie d'initialisation des poids et le prétraitement des données.
* **Pistes d'amélioration :** Le modèle sature vite (dès l'époque 5). Pour repousser cette limite lors de nos prochaines expériences, nous pourrons tester :
  1. Une augmentation du taux de **Dropout** (ex: `0.3` ou `0.4`) pour limiter le surapprentissage.
  2. L'ajout de régularisation L2 via le paramètre **`weight_decay`** dans l'optimiseur Adam.
  3. Une diminution du taux d'apprentissage (**Learning Rate**) à `5e-4` pour stabiliser la convergence.


### Expérience 2 — MLP Custom + Xavier + normalisation

In [108]:
set_seed(SEED)

model_custom_xavier = MLPCustom(input_dim)
init_weights(model_custom_xavier, strategy="xavier")

model_custom_xavier, history_custom_xavier = train_model(
    model_custom_xavier,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device
)

histories["custom_xavier_scaled"] = history_custom_xavier
models["custom_xavier_scaled"] = model_custom_xavier

Epoch 01/20 | Train Loss: 0.3644 | Val Loss: 0.3288 | Train Acc: 0.8260 | Val Acc: 0.8389
Epoch 02/20 | Train Loss: 0.3225 | Val Loss: 0.3219 | Train Acc: 0.8519 | Val Acc: 0.8417
Epoch 03/20 | Train Loss: 0.3158 | Val Loss: 0.3220 | Train Acc: 0.8538 | Val Acc: 0.8462
Epoch 04/20 | Train Loss: 0.3133 | Val Loss: 0.3208 | Train Acc: 0.8542 | Val Acc: 0.8455
Epoch 05/20 | Train Loss: 0.3109 | Val Loss: 0.3206 | Train Acc: 0.8557 | Val Acc: 0.8459
Epoch 06/20 | Train Loss: 0.3082 | Val Loss: 0.3221 | Train Acc: 0.8597 | Val Acc: 0.8453
Epoch 07/20 | Train Loss: 0.3067 | Val Loss: 0.3220 | Train Acc: 0.8588 | Val Acc: 0.8459
Epoch 08/20 | Train Loss: 0.3048 | Val Loss: 0.3207 | Train Acc: 0.8583 | Val Acc: 0.8458
Epoch 09/20 | Train Loss: 0.3014 | Val Loss: 0.3210 | Train Acc: 0.8603 | Val Acc: 0.8453
Epoch 10/20 | Train Loss: 0.2998 | Val Loss: 0.3230 | Train Acc: 0.8616 | Val Acc: 0.8449
Epoch 11/20 | Train Loss: 0.2990 | Val Loss: 0.3235 | Train Acc: 0.8626 | Val Acc: 0.8462
Epoch 12/2

### Expérience 2 : Comparaison Sequential vs MLPCustom (Classe)

#### 1. Constat majeur : Identité stricte des résultats
L'analyse des métriques de l'Expérience 2 (`MLPCustom`) révèle des valeurs **strictement identiques** au millième près à celles obtenues lors de l'Expérience 1 (`build_mlp_sequential`) :
* **Meilleure Époque :** Époque 05 (`Val Loss : 0.3206` | `Val Acc : 84.59%`)
* **Époque Finale (20) :** `Train Loss : 0.2841` | `Val Loss : 0.3295` | `Train Acc : 86.86%` | `Val Acc : 84.68%`

#### 2. Conclusion théorique et technique
Ce résultat valide expérimentalement un concept fondamental de PyTorch : **la syntaxe d'écriture n'influence pas la performance mathématique du modèle.** 

* Qu'on utilise l'empilement rigide `nn.Sequential` ou la structure orientée objet `nn.Module` (avec les fonctions `__init__` et `forward`), l'architecture interne générée sous le capot est rigoureusement la même.
* Le graphe de calcul et le nombre de paramètres à optimiser étant identiques, l'apprentissage produit les mêmes trajectoires de convergence.

#### 3. Conclusion pour l utilisation des deux approches :
* **Quand utiliser l'approche Sequential ?** Pour des prototypes rapides et des réseaux simples en ligne droite (comme notre architecture actuelle).
* **Quand privilégier l'approche Custom ?** Dès que le projet nécessitera des architectures avancées (connexions résiduelles, branches parallèles ou injection de conditions dynamiques dans la méthode `forward`).


### Expérience 3 — MLP Custom + gaussienne

In [109]:
set_seed(SEED)

model_custom_normal = MLPCustom(input_dim)
init_weights(model_custom_normal, strategy="normal")

model_custom_normal, history_custom_normal = train_model(
    model_custom_normal,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device
)

histories["custom_normal_scaled"] = history_custom_normal
models["custom_normal_scaled"] = model_custom_normal

Epoch 01/20 | Train Loss: 0.3787 | Val Loss: 0.3315 | Train Acc: 0.8291 | Val Acc: 0.8390
Epoch 02/20 | Train Loss: 0.3225 | Val Loss: 0.3240 | Train Acc: 0.8511 | Val Acc: 0.8411
Epoch 03/20 | Train Loss: 0.3183 | Val Loss: 0.3224 | Train Acc: 0.8526 | Val Acc: 0.8465
Epoch 04/20 | Train Loss: 0.3155 | Val Loss: 0.3217 | Train Acc: 0.8547 | Val Acc: 0.8448
Epoch 05/20 | Train Loss: 0.3142 | Val Loss: 0.3218 | Train Acc: 0.8558 | Val Acc: 0.8453
Epoch 06/20 | Train Loss: 0.3118 | Val Loss: 0.3228 | Train Acc: 0.8578 | Val Acc: 0.8436
Epoch 07/20 | Train Loss: 0.3109 | Val Loss: 0.3222 | Train Acc: 0.8559 | Val Acc: 0.8437
Epoch 08/20 | Train Loss: 0.3091 | Val Loss: 0.3226 | Train Acc: 0.8570 | Val Acc: 0.8436
Epoch 09/20 | Train Loss: 0.3083 | Val Loss: 0.3211 | Train Acc: 0.8575 | Val Acc: 0.8430
Epoch 10/20 | Train Loss: 0.3064 | Val Loss: 0.3221 | Train Acc: 0.8583 | Val Acc: 0.8451
Epoch 11/20 | Train Loss: 0.3049 | Val Loss: 0.3235 | Train Acc: 0.8599 | Val Acc: 0.8451
Epoch 12/2

### 📉 Expérience 3 : MLPCustom avec Stratégie d'Initialisation Normale

#### 1. Analyse des métriques de convergence
L'application d'une initialisation normale des poids ($\mu = 0.0$, $\sigma = 0.01$) montre une dynamique d'apprentissage légèrement différente de celle de la stratégie Xavier :
* **Démarrage de l'apprentissage :** Le modèle démarre avec une perte initiale plus élevée à l'Époque 01 (`Train Loss : 0.3787` contre `0.3644` pour Xavier), ce qui prouve que l'ajustement initial est un peu moins optimal pour propager les signaux dès le premier lot.
* **Comportement face au surapprentissage :** La stratégie Normale freine la convergence agressive sur le jeu d'entraînement. À l'Époque 20, la `Train Loss` n'est qu'à `0.2938` (contre `0.2841` pour Xavier). De ce fait, la `Val Loss` reste beaucoup plus stable en fin de course et ne subit pas la même remontée brutale.

#### 2. Restauration du meilleur modèle
Le point d'inflexion où le modèle commence à saturer est repoussé. Le système de sauvegarde identifie l'optimum à l'**Époque 15** :
* **Point de checkpoint :** `Val Loss : 0.3208` | `Val Acc : 84.51%`
* **Précision maximale absolue (Validation) :** `84.65%` (atteinte à l'Époque 03).

#### 3. Conclusion comparative (Xavier vs Normal)
Bien que la trajectoire de la perte soit plus stable et mieux régulée en fin d'entraînement avec l'initialisation Normale, le score de généralisation final reste équivalent à celui de Xavier (~84.5% - 84.6%). L'écart-type de 0.01 sélectionné manuellement s'avère être un choix robuste pour cette taille de réseau, mais il nécessite plus d'époques pour extraire le plein potentiel du modèle par rapport à la formule mathématique auto-adaptative de Xavier.


### Expérience 4 — MLP Custom + constante

In [111]:
set_seed(SEED)

model_custom_constant = MLPCustom(input_dim)
init_weights(model_custom_constant, strategy="constant")

model_custom_constant, history_custom_constant = train_model(
    model_custom_constant,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device
)

histories["custom_constant_scaled"] = history_custom_constant
models["custom_constant_scaled"] = model_custom_constant

Epoch 01/20 | Train Loss: 30765.7063 | Val Loss: 19548.4392 | Train Acc: 0.2508 | Val Acc: 0.2589
Epoch 02/20 | Train Loss: 13687.3997 | Val Loss: 8841.9797 | Train Acc: 0.2757 | Val Acc: 0.2982
Epoch 03/20 | Train Loss: 6388.0005 | Val Loss: 4231.3081 | Train Acc: 0.3224 | Val Acc: 0.3491
Epoch 04/20 | Train Loss: 3173.0716 | Val Loss: 2164.7382 | Train Acc: 0.3830 | Val Acc: 0.4150
Epoch 05/20 | Train Loss: 1683.6233 | Val Loss: 1180.4200 | Train Acc: 0.4468 | Val Acc: 0.4771
Epoch 06/20 | Train Loss: 950.0590 | Val Loss: 679.2575 | Train Acc: 0.5045 | Val Acc: 0.5282
Epoch 07/20 | Train Loss: 561.7357 | Val Loss: 408.3181 | Train Acc: 0.5482 | Val Acc: 0.5778
Epoch 08/20 | Train Loss: 346.6287 | Val Loss: 254.0354 | Train Acc: 0.5852 | Val Acc: 0.6133
Epoch 09/20 | Train Loss: 219.3335 | Val Loss: 163.3507 | Train Acc: 0.6178 | Val Acc: 0.6400
Epoch 10/20 | Train Loss: 143.0937 | Val Loss: 107.0825 | Train Acc: 0.6433 | Val Acc: 0.6680
Epoch 11/20 | Train Loss: 94.6595 | Val Loss: 7

### Expérience 4 : MLPCustom avec Stratégie d'Initialisation Constante (= 1.0)

#### 1. Constat majeur : Une anomalie mathématique brutale
Cette expérience met en évidence l'échec critique de l'initialisation constante des poids, caractérisée par des métriques hors-normes au départ :
* **Explosion de la Perte (*Loss*) :** À l'Époque 01, la `Train Loss` explose à un niveau astronomique de **`30 765.7063`** (comparé à `0.36` pour Xavier). 
* **Effondrement de la Précision (*Accuracy*) :** Le modèle démarre avec une exactitude de seulement **`25.08%`**, ce qui correspond au niveau d'une prédiction uniforme sur la classe minoritaire du dataset.

#### 2. Explication théorique : La rupture de symétrie et l'explosion des gradients
Ce phénomène s'explique par deux concepts fondamentaux du Deep Learning :
1. **La Symétrie des Neurones :** En attribuant la valeur identique de `1.0` à tous les poids, tous les neurones d'une couche cachée calculent rigoureusement la même fonction. Lors de la rétropropagation (*backpropagation*), ils reçoivent tous exactement le même gradient. Le réseau perd sa capacité à apprendre des caractéristiques différentes (incapacité à briser la symétrie).
2. **L'Explosion des Activations :** Sans pondération distribuée (comme le propose Xavier ou la loi Normale), les valeurs numériques sont amplifiées de manière exponentielle à travers les couches linéaires successives, saturant totalement la fonction de perte en sortie.

#### 3. Résilience de l'optimiseur Adam et conclusion
Au fil des époques, l'optimiseur **Adam** parvient à corriger progressivement le tir en réduisant agressivement la magnitude des poids (la Loss chute à `1.2962` à l'Époque 20). 

Cependant, le modèle reste piégé dans un minimum local médiocre. Sa précision finale plafonne à **`78.89%`**, soit une perte nette de performance de près de **6%** par rapport aux initialisations aléatoires. 

**Conclusion du protocole d'expériences :** L'initialisation constante à 1.0 est à proscrire définitivement. La stratégie **Xavier** (ou Normale bien calibrée) reste indispensable pour permettre la convergence rapide et performante d'un Perceptron Multicouche.


### Expérience 5 — MLP Custom + Xavier sans normalisation

In [112]:
input_dim_no_scale = X_train_no_scale.shape[1]

set_seed(SEED)

model_custom_xavier_no_scale = MLPCustom(input_dim_no_scale)
init_weights(model_custom_xavier_no_scale, strategy="xavier")

model_custom_xavier_no_scale, history_custom_xavier_no_scale = train_model(
    model_custom_xavier_no_scale,
    train_loader_no_scale,
    val_loader_no_scale,
    epochs=EPOCHS,
    lr=LR,
    device=device
)

histories["custom_xavier_no_scale"] = history_custom_xavier_no_scale
models["custom_xavier_no_scale"] = model_custom_xavier_no_scale

Epoch 01/20 | Train Loss: 574.3931 | Val Loss: 1.3176 | Train Acc: 0.6423 | Val Acc: 0.2478
Epoch 02/20 | Train Loss: 6.4882 | Val Loss: 0.6168 | Train Acc: 0.7146 | Val Acc: 0.7545
Epoch 03/20 | Train Loss: 2.6740 | Val Loss: 0.5721 | Train Acc: 0.7381 | Val Acc: 0.7544
Epoch 04/20 | Train Loss: 1.3026 | Val Loss: 0.5593 | Train Acc: 0.7450 | Val Acc: 0.7539
Epoch 05/20 | Train Loss: 0.9934 | Val Loss: 0.5762 | Train Acc: 0.7485 | Val Acc: 0.7539
Epoch 06/20 | Train Loss: 0.9884 | Val Loss: 0.5659 | Train Acc: 0.7518 | Val Acc: 0.7539
Epoch 07/20 | Train Loss: 0.7084 | Val Loss: 0.5623 | Train Acc: 0.7536 | Val Acc: 0.7528
Epoch 08/20 | Train Loss: 0.7672 | Val Loss: 0.5602 | Train Acc: 0.7540 | Val Acc: 0.7528
Epoch 09/20 | Train Loss: 0.6479 | Val Loss: 0.5601 | Train Acc: 0.7529 | Val Acc: 0.7523
Epoch 10/20 | Train Loss: 0.7269 | Val Loss: 0.5592 | Train Acc: 0.7534 | Val Acc: 0.7528
Epoch 11/20 | Train Loss: 0.7591 | Val Loss: 0.5577 | Train Acc: 0.7538 | Val Acc: 0.7538
Epoch 12

### Expérience 5 : MLPCustom avec Initialisation Xavier Sans Normalisation des Données

#### 1. Constat majeur : Perte de performance radicale
Le retrait du prétraitement par mise à l'échelle (`StandardScaler`) engendre une dégradation immédiate et sévère des capacités du réseau de neurones :
* **Instabilité initiale :** À l'Époque 01, la `Train Loss` démarre à un niveau très élevé de **`574.3931`** tandis que la `Val Acc` s'effondre à **`24.78%`**, témoignant d'une explosion numérique due à la disparité des échelles des variables d'entrée.
* **Plafonnement des performances :** Dès l'Époque 02, l'optimiseur stabilise le modèle, mais celui-ci s'enferme définitivement dans un plateau médiocre. La précision stagne autour de **`75.5% - 75.6%`** jusqu'à la fin de l'entraînement.

#### 2. Explication théorique : La domination des variables à forte magnitude
Les réseaux de neurones sont extrêmement sensibles à l'échelle des descripteurs (*features*) :
1. **Étouffement des petits signaux :** Sans normalisation, les variables possédant de grandes valeurs numériques (comme le gain en capital ou l'âge) génèrent des gradients disproportionnés lors de la rétropropagation. Elles masquent et étouffent totalement l'influence des variables binaires ou à faible amplitude (comme le sexe, l'éducation ou l'origine).
2. **Stratégie de prédiction paresseuse :** Ne parvenant pas à combiner harmonieusement toutes les fonctionnalités, le modèle converge vers un minimum local "facile" : prédire systématiquement la classe majoritaire du dataset (qui représente historiquement environ 75% des données).

#### 3. Conclusion du protocole
Cette expérience démontre que même l'utilisation d'une initialisation de pointe comme **Xavier** ne peut compenser l'absence de normalisation. Le *Feature Scaling* via un `StandardScaler` reste une étape **non-négociable** et obligatoire en Deep Learning pour garantir que chaque variable contribue équitablement à la décision du modèle.
